In [ ]:
# BLOQUE 1 — Configuración del snapshot GTFS a cargar en Bronze

catalog = "workspace"
schema = "gtfs_bogota"
staging_volume_path = f"/Volumes/{catalog}/{schema}/staging_gtfs"

feed_version = "2025-10-28"
snapshot_sha256 = "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952"

staging_snapshot_path = (
    f"{staging_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}"
)

print(f"Snapshot de Staging: {staging_snapshot_path}")
display(dbutils.fs.ls(staging_snapshot_path))

In [ ]:
# BLOQUE 2 — Contrato de entidades GTFS operacionales

# Estas entidades tienen mayor volumen y conectan viajes, paradas y geometrías.
operational_entities = [
    {
        "file_name": "shapes.txt",
        "table_name": "bronze_gtfs_shapes",
        "required_columns": ["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
    },
    {
        "file_name": "stop_times.txt",
        "table_name": "bronze_gtfs_stop_times",
        "required_columns": ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
    },
]

In [ ]:
# BLOQUE 3 — Funciones reutilizables de validación y escritura Bronze

import csv

from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType,
)

def read_gtfs_header(source_path: str) -> list[str]:
    """Lee el encabezado UTF-8 y evita contratos ambiguos."""
    with open(source_path, "r", encoding="utf-8-sig", newline="") as source_file:
        header = next(csv.reader(source_file), None)

    if not header or any(not column.strip() for column in header):
        raise RuntimeError(f"Encabezado inválido o vacío en {source_path}.")
    if len(header) != len(set(header)):
        raise RuntimeError(f"El archivo tiene columnas duplicadas: {source_path}.")

    return header


def load_entity_to_bronze(entity: dict) -> tuple:
    """Carga una entidad por hash, sin duplicarla en reintentos."""
    source_path = f"{staging_snapshot_path}/{entity['file_name']}"
    table_name = f"{catalog}.{schema}.{entity['table_name']}"
    source_columns = read_gtfs_header(source_path)

    missing_columns = sorted(set(entity['required_columns']) - set(source_columns))
    if missing_columns:
        raise RuntimeError(
            f"Contrato GTFS incumplido en {entity['file_name']}. "
            f"Faltan columnas: {', '.join(missing_columns)}"
        )

    source_schema = StructType([
        StructField(column, StringType(), True)
        for column in source_columns
    ])

    source_df = (
        spark.read
        .schema(source_schema)
        .option("header", True)
        .option("mode", "FAILFAST")
        .csv(source_path)
    )

    # Persistimos temporalmente para no leer dos veces los archivos más grandes.
    bronze_df = (
        source_df
        .withColumn("source_feed_version", F.lit(feed_version))
        .withColumn("source_sha256", F.lit(snapshot_sha256))
        .withColumn("source_file_path", F.lit(source_path))
        .withColumn("bronze_loaded_at_utc", F.current_timestamp())
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    try:
        source_row_count = bronze_df.count()

        if not spark.catalog.tableExists(table_name):
            (
                bronze_df.limit(0)
                .write
                .format("delta")
                .saveAsTable(table_name)
            )

        snapshot_already_loaded = (
            spark.table(table_name)
            .filter(F.col("source_sha256") == snapshot_sha256)
            .limit(1)
            .count() > 0
        )

        if not snapshot_already_loaded:
            (
                bronze_df
                .write
                .format("delta")
                .mode("append")
                .saveAsTable(table_name)
            )

        return (
            entity['file_name'],
            table_name,
            source_row_count,
            snapshot_already_loaded,
        )
    finally:
        bronze_df.unpersist()


load_result_schema = StructType([
    StructField("source_file_name", StringType(), False),
    StructField("bronze_table_name", StringType(), False),
    StructField("source_row_count", LongType(), False),
    StructField("snapshot_already_loaded", BooleanType(), False),
])

In [ ]:
# BLOQUE 4 — Carga idempotente de shapes.txt

shapes_result = load_entity_to_bronze(operational_entities[0])

display(spark.createDataFrame([shapes_result], schema=load_result_schema))

In [ ]:
# BLOQUE 5 — Carga idempotente de stop_times.txt

# stop_times conecta los viajes con las paradas y es la entidad más relevante para Silver.
stop_times_result = load_entity_to_bronze(operational_entities[1])

display(spark.createDataFrame([stop_times_result], schema=load_result_schema))

In [ ]:
# BLOQUE 6 — Verificación de las entidades operacionales persistidas

bronze_stop_times_table = f"{catalog}.{schema}.bronze_gtfs_stop_times"

display(
    spark.table(bronze_stop_times_table)
    .filter(F.col("source_sha256") == snapshot_sha256)
    .orderBy("trip_id", "stop_sequence")
    .limit(20)
)

display(spark.sql(f"describe detail {bronze_stop_times_table}"))